# ViSEF — Colab T4 (seed 456)

Notebook độc lập để chạy seed còn lại sau hai seed 42 và 123 trên Kaggle. Cần chọn GPU T4 trong **Runtime → Change runtime type**. Tải `kaggle_seeds_42_123.zip` từ Output Kaggle notebook lên Colab khi được nhắc; notebook sẽ hợp nhất runs và partitions trước khi chạy seed 456.

Mã nguồn được clone từ GitHub; dữ liệu MRI được chuẩn bị như notebook gốc.

In [ ]:
!pip -q install torch torchvision numpy Pillow matplotlib opacus
!git clone --depth 1 https://github.com/nguyenquocthanh1213/KHKT.git /content/KHKT
%cd /content/KHKT
import torch, os, sys, json, subprocess, zipfile, shutil, time
from pathlib import Path
from google.colab import files
print('torch',torch.__version__,'| CUDA',torch.cuda.is_available(),'| GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
assert torch.cuda.is_available(), 'Hãy bật GPU T4 trong Runtime.'


In [ ]:
# Upload artifact Kaggle (chứa runs seed 42/123). Bỏ qua nếu chạy độc lập để chỉ tạo seed 456.
up = files.upload()
for name in up:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z: z.extractall('/content/KHKT')
        print('Đã giải nén',name)


In [ ]:
# Tải và chuẩn hóa dataset, tạo partition chỉ khi chưa nhận từ Kaggle.
%cd /content/KHKT
!sed -i 's/\r$//' src/prepare_data.sh
!bash src/prepare_data.sh
DATA_ROOT=Path('brain-tumor-mri-dataset')
assert (DATA_ROOT/'Training').is_dir() and (DATA_ROOT/'Testing').is_dir()
if not Path('partitions/global_val.json').exists():
    subprocess.run([sys.executable,'src/partition.py','--data_root',str(DATA_ROOT),'--out_dir','partitions',
                    '--clients','A:glioma','B:meningioma','C:notumor','D:pituitary',
                    '--samples_per_client','1200','--dominant_frac','0.7','--val_per_class','100','--seed','42'],check=True)


In [ ]:
QUICK_MODE=False
SEED=456
ROUNDS=20 if not QUICK_MODE else 5
EPOCHS,BATCH,LR=3,32,1e-4
DOM_FRAC,TRAIN_MODE=0.7,'linear'
CONFIGS={
 'fedavg':dict(mu=0.0,dp=False), 'fedprox':dict(mu=0.01,dp=False),
 'fedprox_dp':dict(mu=0.01,dp=True), 'fedprox_dp_autoclip':dict(mu=0.01,dp=True,auto_clip=True),
}
def sh(cmd):
 print('\n$', ' '.join(map(str,cmd)))
 subprocess.run([str(x) for x in cmd],check=True)
def pick_best_round(run_dir):
 val=json.loads((Path(run_dir)/'eval/eval_val.json').read_text())['results']
 srv=[r for r in val if r['kind']=='server' and r['round']>0]
 return max(srv,key=lambda r:r['macro_f1'])['round'] if srv else None


In [ ]:
t0=time.time()
for name,cfg in CONFIGS.items():
 tag=f'p4_linear_{name}_seed{SEED}'; run_dir=Path('runs')/tag
 if (run_dir/'eval/eval_test.json').exists():
  print('[SKIP]',tag,'done'); continue
 cmd=[sys.executable,'src/run_simulation.py','--data_root',str(DATA_ROOT),'--tag',tag,
      '--partition_dir','partitions','--rounds',str(ROUNDS),'--epochs',str(EPOCHS),
      '--batch_size',str(BATCH),'--lr',str(LR),'--train_mode',TRAIN_MODE,'--mu',str(cfg['mu']),
      '--dominant_frac',str(DOM_FRAC),'--seed',str(SEED),'--skip_eval','--device','auto']
 if cfg['dp']:
  cmd += ['--dp','--noise_multiplier','1.0','--max_grad_norm','1.0','--verify_dp']
  if cfg.get('auto_clip'): cmd += ['--auto_clip']
 sh(cmd)
 sh([sys.executable,'src/evaluate.py','--run_dir',str(run_dir),'--split','val','--partition_dir','partitions',
     '--all_rounds','--device','auto','--no_plots'])
 best=pick_best_round(run_dir); print('[Pick]',tag,'best validation round:',best)
 sh([sys.executable,'src/evaluate.py','--run_dir',str(run_dir),'--split','test','--data_root',str(DATA_ROOT),
     '--server_rounds',str(best),'--client_rounds','','--device','auto','--no_plots'])
print(f'Hoàn tất seed 456 trong {(time.time()-t0)/3600:.2f} giờ.')


In [ ]:
# Tạo artifact Colab gồm seed 456 và bộ runs đã import từ Kaggle để tải xuống.
archive=shutil.make_archive('/content/ViSEF_runs_42_123_456','zip',root_dir='/content/KHKT',base_dir='runs')
print('Artifact:',archive)
files.download(archive)


## Tổng hợp cuối

Sau khi tải artifact từ Colab, giải nén vào cùng thư mục dự án với mã nguồn để có `runs/p4_linear_*_seed{42,123,456}`. Chạy `python src/summarize_results.py` để tổng hợp đủ ba seed. Nếu không tải artifact Kaggle trước, notebook vẫn chạy seed 456 nhưng chưa có đủ dữ liệu để tính mean ± std.